# Notebook 03 — Knowledge Distillation in Practice (Teacher → Student) for Text Classification  
**GPU-first (Colab), with CPU fallback** — realistic workflow + metrics (**accuracy, latency, memory**)

In this notebook you will implement a **realistic distillation pipeline**:

1. Pick a **strong teacher** model already fine-tuned for sentiment classification.
2. Pick a **smaller student** model.
3. Train the student with **hard labels + soft targets** from the teacher:
   - hard-label loss: cross-entropy on ground-truth labels
   - soft-target loss: KL divergence between teacher and student distributions (with temperature)
4. Evaluate and compare:
   - **Accuracy** on the validation set (teacher vs student baseline vs distilled student)
   - **Latency** (ms / example) on CPU and GPU
   - **Memory** (parameter count, model size, peak GPU memory)

We use the **SST-2** sentiment dataset (GLUE) because it is a standard NLP benchmark and runs quickly.

> This notebook is intentionally detailed. Each cell explains what it does and why.


## 1) Install dependencies

We use:
- `transformers`, `accelerate`: modeling and training
- `datasets`, `evaluate`: dataset + accuracy metric
- `torch`: backend
- `psutil`: measure CPU memory (optional but useful)


In [1]:
import sys, subprocess

def pip_install(pkgs):
    print("Installing:", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install([
    "numpy>=1.24",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "datasets>=2.18",
    "evaluate>=0.4.2",
    "transformers>=4.41",
    "accelerate>=0.30",
    "psutil>=5.9",
])

print("Done. If imports fail later, try Runtime -> Restart runtime.")


Installing: ['numpy>=1.24', 'pandas>=2.0', 'matplotlib>=3.7', 'datasets>=2.18', 'evaluate>=0.4.2', 'transformers>=4.41', 'accelerate>=0.30', 'psutil>=5.9']
Done. If imports fail later, try Runtime -> Restart runtime.


## 2) Imports + seed + device selection (GPU-first)

We set:
- deterministic seeds (as much as possible)
- `DEVICE` = CUDA if available else CPU

We also print GPU details if running on CUDA.


In [2]:
import os, time, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import psutil

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def get_device():
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

DEVICE = get_device()

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU RAM: {props.total_memory/(1024**3):.1f} GB")


Device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU RAM: 79.3 GB


## 3) Choose teacher and student models (realistic setup)

We want a **teacher** that is strong *and already fine-tuned* for SST-2.

- Teacher: `textattack/roberta-base-SST-2`  
  (RoBERTa-base fine-tuned for SST-2; strong accuracy)

We want a smaller **student**:
- Student: `distilbert-base-uncased`  
  (smaller and faster, but not fine-tuned)

We will train **two students**:
1. **Baseline student**: fine-tune with hard labels only
2. **Distilled student**: fine-tune with hard labels + teacher soft targets

This gives a fair comparison.


In [3]:
TEACHER_NAME = "textattack/roberta-base-SST-2"
STUDENT_NAME = "distilbert-base-uncased"

print("Teacher:", TEACHER_NAME)
print("Student:", STUDENT_NAME)


Teacher: textattack/roberta-base-SST-2
Student: distilbert-base-uncased


## 4) Load dataset: GLUE SST-2

SST-2 is a binary sentiment classification dataset:
- label 0: negative
- label 1: positive

We will:
- load train + validation splits
- optionally take a subset for faster experimentation


In [4]:
from datasets import load_dataset

ds = load_dataset("glue", "sst2")
print(ds)

train_ds = ds["train"]
val_ds = ds["validation"]

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("Example:", train_ds[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
Train size: 67349
Val size: 872
Example: {'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


### 4.1) Use a training subset (recommended for notebook speed)

Distillation can be compute-heavy because it involves teacher signals.
To keep the notebook practical, we use a subset by default.

If you have a strong GPU, you can increase `TRAIN_SUBSET` (or set it to `None` to use full training set).


In [5]:
TRAIN_SUBSET = 20000   # set to None for full train set
VAL_SUBSET = None      # usually keep full validation

if TRAIN_SUBSET is not None:
    train_ds = train_ds.shuffle(seed=SEED).select(range(TRAIN_SUBSET))
if VAL_SUBSET is not None:
    val_ds = val_ds.select(range(VAL_SUBSET))

print("Train used:", len(train_ds))
print("Val used:", len(val_ds))


Train used: 20000
Val used: 872


## 5) Tokenization

Teacher and student have **different tokenizers**.
We tokenize twice:

- Teacher tokenizer: RoBERTa
- Student tokenizer: DistilBERT


In [6]:
from transformers import AutoTokenizer

teacher_tok = AutoTokenizer.from_pretrained(TEACHER_NAME, use_fast=True)
student_tok = AutoTokenizer.from_pretrained(STUDENT_NAME, use_fast=True)

MAX_LEN = 128

def tok_teacher(batch):
    return teacher_tok(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN)

def tok_student(batch):
    return student_tok(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN)

train_teacher_tok = train_ds.map(tok_teacher, batched=True, remove_columns=train_ds.column_names)
val_teacher_tok = val_ds.map(tok_teacher, batched=True, remove_columns=val_ds.column_names)

train_student_tok = train_ds.map(tok_student, batched=True, remove_columns=train_ds.column_names)
val_student_tok = val_ds.map(tok_student, batched=True, remove_columns=val_ds.column_names)

# Add labels
train_student_tok = train_student_tok.add_column("labels", train_ds["label"])
val_student_tok = val_student_tok.add_column("labels", val_ds["label"])

train_teacher_tok = train_teacher_tok.add_column("labels", train_ds["label"])
val_teacher_tok = val_teacher_tok.add_column("labels", val_ds["label"])

print(train_student_tok)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 20000
})


## 6) Load models + helpers

- Teacher is used in eval mode (no gradients).
- Two students:
  - baseline student (hard labels only)
  - distilled student (hard labels + teacher logits)

We also define helpers for parameter counting and memory sizing.


In [7]:
from transformers import AutoModelForSequenceClassification

teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_NAME, num_labels=2).to(DEVICE)
teacher.eval()

student_base = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=2).to(DEVICE)
student_kd   = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=2).to(DEVICE)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(model):
    total_bytes = 0
    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()
    return total_bytes / (1024**2)

print("Teacher params:", f"{count_params(teacher)/1e6:.1f}M", "≈", f"{model_size_mb(teacher):.1f} MB")
print("Student params:", f"{count_params(student_base)/1e6:.1f}M", "≈", f"{model_size_mb(student_base):.1f} MB")


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at textattack/roberta-base-SST-2 were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Teacher params: 124.6M ≈ 475.5 MB
Student params: 67.0M ≈ 255.4 MB


## 7) Evaluate teacher (upper bound)

We compute validation accuracy for the teacher.


In [8]:
import evaluate
from transformers import DataCollatorWithPadding

acc_metric = evaluate.load("accuracy")
teacher_collator = DataCollatorWithPadding(tokenizer=teacher_tok)

def eval_model_accuracy(model, dataset_tok, collator, batch_size=64):
    model.eval()
    metric = evaluate.load("accuracy")
    loader = torch.utils.data.DataLoader(dataset_tok, batch_size=batch_size, shuffle=False, collate_fn=collator)

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.no_grad():
            logits = model(**batch).logits
        preds = torch.argmax(logits, dim=-1)
        metric.add_batch(predictions=preds.cpu().numpy(), references=labels.cpu().numpy())

    return metric.compute()["accuracy"]

teacher_acc = eval_model_accuracy(
    teacher, val_teacher_tok, teacher_collator,
    batch_size=128 if DEVICE.type=="cuda" else 32
)
teacher_acc


0.9403669724770642

## 8) Precompute teacher logits (offline distillation)

A realistic setup often uses **offline teacher predictions**:
- Run teacher once over the training set
- Store logits as soft targets for student training

This avoids running the teacher on every step (which is expensive).


In [9]:
from tqdm.auto import tqdm

def compute_teacher_logits(teacher_model, dataset_tok, collator, batch_size=128):
    teacher_model.eval()
    loader = torch.utils.data.DataLoader(dataset_tok, batch_size=batch_size, shuffle=False, collate_fn=collator)

    all_logits = []
    for batch in tqdm(loader, desc="Teacher inference"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        _ = batch.pop("labels")
        with torch.no_grad():
            logits = teacher_model(**batch).logits
        all_logits.append(logits.detach().cpu())
    return torch.cat(all_logits, dim=0).numpy()

bs = 256 if DEVICE.type=="cuda" else 32
t0 = time.time()
teacher_logits_train = compute_teacher_logits(teacher, train_teacher_tok, teacher_collator, batch_size=bs)
print("Teacher logits shape:", teacher_logits_train.shape)
print(f"Time: {time.time()-t0:.1f}s")


Teacher inference:   0%|          | 0/79 [00:00<?, ?it/s]

Teacher logits shape: (20000, 2)
Time: 29.6s


### 8.1) Attach teacher logits to the student training dataset

We add a column `teacher_logits`.


In [10]:
train_student_tok = train_student_tok.add_column("teacher_logits", teacher_logits_train.tolist())

print(train_student_tok[0].keys())
print("teacher_logits example:", train_student_tok[0]["teacher_logits"])


dict_keys(['input_ids', 'attention_mask', 'labels', 'teacher_logits'])
teacher_logits example: [-3.777665376663208, 3.832117795944214]


## 9) Define the distillation loss

Total loss:
    loss = alpha * hard_loss + (1 - alpha) * (T^2) * KL(teacher || student)

Where:
- `T` is temperature (softens distributions)
- `alpha` balances hard vs soft supervision


In [16]:
import torch.nn.functional as F
from transformers import Trainer

T = 2.0
ALPHA = 0.5

class DistillationTrainer(Trainer):
    def __init__(self, *args, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        teacher_logits = inputs.pop("teacher_logits", None)

        outputs = model(**inputs)
        student_logits = outputs.logits

        hard_loss = F.cross_entropy(student_logits, labels)

        if teacher_logits is None:
            loss = hard_loss
        else:
            teacher_logits = torch.tensor(teacher_logits, device=student_logits.device, dtype=student_logits.dtype)
            T = self.temperature

            teacher_logp = F.log_softmax(teacher_logits / T, dim=-1)
            student_logp = F.log_softmax(student_logits / T, dim=-1)
            teacher_p = teacher_logp.exp()

            kl = F.kl_div(student_logp, teacher_p, reduction="batchmean")
            loss = self.alpha * hard_loss + (1 - self.alpha) * (T * T) * kl

        return (loss, outputs) if return_outputs else loss

print("DistillationTrainer ready.")

DistillationTrainer ready.


## 10) Training configuration

We will train:
- baseline student (hard labels only)
- distilled student (hard + soft targets)

Default values are notebook-friendly but realistic.


In [18]:
from transformers import TrainingArguments, DataCollatorWithPadding

BATCH_SIZE = 32 if DEVICE.type=="cuda" else 8
NUM_EPOCHS = 2

OUT_BASE = "student_baseline_ckpt"
OUT_KD   = "student_distilled_ckpt"

common_args = dict(
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    report_to=[],
    remove_unused_columns=False, # Necessary to keep teacher_logits
)

base_args = TrainingArguments(output_dir=OUT_BASE, **common_args)
kd_args   = TrainingArguments(output_dir=OUT_KD, **common_args)

student_collator = DataCollatorWithPadding(tokenizer=student_tok)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

print("Batch size:", BATCH_SIZE, "Epochs:", NUM_EPOCHS)

Batch size: 32 Epochs: 2


## 11) Baseline student training (hard labels only)


In [19]:
from transformers import Trainer

baseline_trainer = Trainer(
    model=student_base,
    args=base_args,
    train_dataset=train_student_tok.remove_columns(["teacher_logits"]),
    eval_dataset=val_student_tok,
    data_collator=student_collator,
    compute_metrics=compute_metrics,
)

t0 = time.time()
baseline_trainer.train()
print(f"Baseline training time: {time.time()-t0:.1f}s")

baseline_eval = baseline_trainer.evaluate()
baseline_eval


Epoch,Training Loss,Validation Loss,Accuracy
1,0.105500,0.477870,0.870413
2,0.104500,0.523467,0.880734


Baseline training time: 104.8s


{'eval_loss': 0.5234671831130981,
 'eval_accuracy': 0.8807339449541285,
 'eval_runtime': 0.8046,
 'eval_samples_per_second': 1083.726,
 'eval_steps_per_second': 17.399,
 'epoch': 2.0}

## 12) Distilled student training (hard labels + teacher logits)


In [20]:
# Training the distilled student with the fixed Trainer
kd_trainer = DistillationTrainer(
    model=student_kd,
    args=kd_args,
    train_dataset=train_student_tok,
    eval_dataset=val_student_tok,
    data_collator=student_collator,
    compute_metrics=compute_metrics,
    temperature=T,
    alpha=ALPHA,
)

t0 = time.time()
kd_trainer.train()
print(f"Distillation training time: {time.time()-t0:.1f}s")

kd_eval = kd_trainer.evaluate()
kd_eval

/tmp/ipython-input-2602802799.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  teacher_logits = torch.tensor(teacher_logits, device=student_logits.device, dtype=student_logits.dtype)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.382400,0.442113,0.854358
2,0.227800,0.375617,0.888761


Distillation training time: 105.7s


{'eval_loss': 0.37561658024787903,
 'eval_accuracy': 0.8887614678899083,
 'eval_runtime': 0.8067,
 'eval_samples_per_second': 1080.991,
 'eval_steps_per_second': 17.355,
 'epoch': 2.0}

## 13) Compare accuracy (teacher vs students)


In [21]:
results = pd.DataFrame([
    {"model": "Teacher (RoBERTa SST-2)", "accuracy": teacher_acc},
    {"model": "Student baseline (DistilBERT)", "accuracy": baseline_eval["eval_accuracy"]},
    {"model": "Student distilled (DistilBERT + KD)", "accuracy": kd_eval["eval_accuracy"]},
])
results


,model,accuracy
0,Teacher (RoBERTa SST-2),0.940367
1,Student baseline (DistilBERT),0.880734
2,Student distilled (DistilBERT + KD),0.888761


## 14) Measure inference latency (CPU and GPU)

We measure **ms/example** for:
- teacher
- baseline student
- distilled student

On:
- current device (GPU if available)
- CPU

We keep tokenization out of timing (tokenize once).


In [22]:
from copy import deepcopy

def measure_latency_ms_per_example(model, tokenizer, sentences, device, warmup=3, iters=10):
    model = model.to(device)
    model.eval()

    enc = tokenizer(sentences, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(**enc).logits
        if device.type == "cuda":
            torch.cuda.synchronize()

    start = time.time()
    with torch.no_grad():
        for _ in range(iters):
            _ = model(**enc).logits
        if device.type == "cuda":
            torch.cuda.synchronize()
    elapsed = time.time() - start

    total_examples = iters * len(sentences)
    return (elapsed / total_examples) * 1000.0

LAT_N = 64
sent_batch = [val_ds[i]["sentence"] for i in range(min(LAT_N, len(val_ds)))]

lat_rows = []
lat_rows.append({"model": "Teacher", "device": str(DEVICE),
                 "ms_per_example": measure_latency_ms_per_example(deepcopy(teacher), teacher_tok, sent_batch, DEVICE)})
lat_rows.append({"model": "Student baseline", "device": str(DEVICE),
                 "ms_per_example": measure_latency_ms_per_example(deepcopy(student_base), student_tok, sent_batch, DEVICE)})
lat_rows.append({"model": "Student distilled", "device": str(DEVICE),
                 "ms_per_example": measure_latency_ms_per_example(deepcopy(student_kd), student_tok, sent_batch, DEVICE)})

cpu = torch.device("cpu")
lat_rows.append({"model": "Teacher", "device": "cpu",
                 "ms_per_example": measure_latency_ms_per_example(deepcopy(teacher).to("cpu"), teacher_tok, sent_batch, cpu)})
lat_rows.append({"model": "Student baseline", "device": "cpu",
                 "ms_per_example": measure_latency_ms_per_example(deepcopy(student_base).to("cpu"), student_tok, sent_batch, cpu)})
lat_rows.append({"model": "Student distilled", "device": "cpu",
                 "ms_per_example": measure_latency_ms_per_example(deepcopy(student_kd).to("cpu"), student_tok, sent_batch, cpu)})

lat_df = pd.DataFrame(lat_rows)
lat_df


,model,device,ms_per_example
0,Teacher,cuda,1.362849
1,Student baseline,cuda,0.683683
2,Student distilled,cuda,0.683533
3,Teacher,cpu,44.067940
4,Student baseline,cpu,21.164163
5,Student distilled,cpu,21.458502


## 15) Measure memory: parameters + model size + peak GPU forward memory

We report:
- parameter count (M)
- approximate parameter memory (MB)
- peak GPU memory (MB) for one forward pass (if CUDA)


In [23]:
def peak_gpu_mem_mb_forward(model, tokenizer, sentences, device):
    if device.type != "cuda":
        return None
    torch.cuda.reset_peak_memory_stats()
    model = model.to(device)
    model.eval()
    enc = tokenizer(sentences, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        _ = model(**enc).logits
        torch.cuda.synchronize()
    return float(torch.cuda.max_memory_allocated() / (1024**2))

mem_rows = []
for name, model, tok in [
    ("Teacher", teacher, teacher_tok),
    ("Student baseline", student_base, student_tok),
    ("Student distilled", student_kd, student_tok),
]:
    mem_rows.append({
        "model": name,
        "params_M": count_params(model)/1e6,
        "param_size_MB": model_size_mb(model),
        "peak_gpu_MB_forward": peak_gpu_mem_mb_forward(model, tok, sent_batch, DEVICE),
    })

mem_df = pd.DataFrame(mem_rows)
mem_df


,model,params_M,param_size_MB,peak_gpu_MB_forward
0,Teacher,124.64717,475.491219,2877.281738
1,Student baseline,66.95501,255.413094,2877.281738
2,Student distilled,66.95501,255.413094,2877.281738


## 16) Final comparison table (accuracy + latency + memory)


In [24]:
acc_map = {
    "Teacher": teacher_acc,
    "Student baseline": baseline_eval["eval_accuracy"],
    "Student distilled": kd_eval["eval_accuracy"],
}

final_rows = []
for m in ["Teacher", "Student baseline", "Student distilled"]:
    m_mem = mem_df[mem_df["model"] == m].iloc[0].to_dict()
    final_rows.append({
        "model": m,
        "accuracy": acc_map[m],
        "params_M": m_mem["params_M"],
        "param_size_MB": m_mem["param_size_MB"],
        "gpu_ms_per_ex": float(lat_df[(lat_df["model"] == m) & (lat_df["device"] == str(DEVICE))]["ms_per_example"].iloc[0]),
        "cpu_ms_per_ex": float(lat_df[(lat_df["model"] == m) & (lat_df["device"] == "cpu")]["ms_per_example"].iloc[0]),
        "peak_gpu_MB_forward": m_mem["peak_gpu_MB_forward"],
    })

final_df = pd.DataFrame(final_rows)
final_df


,model,accuracy,params_M,param_size_MB,gpu_ms_per_ex,cpu_ms_per_ex,peak_gpu_MB_forward
0,Teacher,0.940367,124.64717,475.491219,1.362849,44.067940,2877.281738
1,Student baseline,0.880734,66.95501,255.413094,0.683683,21.164163,2877.281738
2,Student distilled,0.888761,66.95501,255.413094,0.683533,21.458502,2877.281738


## 78) Optional extensions

1. **Online distillation**: teacher runs every step.
2. **Intermediate-layer distillation**: align hidden states/attention.
3. **Multi-objective model selection**: pick student by (accuracy, latency, size).
4. **Distillation + quantization**: quantize the distilled student (INT8/INT4).
